In [2]:
import os
import yaml
import glob
import warnings; warnings.filterwarnings('ignore')

from dotenv import load_dotenv
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain

In [3]:
load_dotenv(dotenv_path="../.env") 

with open("../config/config.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

env_var_name = config['api_keys']['google_gemini_env_name']
gemini_api_key = os.getenv(env_var_name)

if gemini_api_key:
    os.environ["GOOGLE_API_KEY"] = gemini_api_key
    print("✅ Gemini API 키가 성공적으로 로드되었습니다.")
else:
    print("❌ API 키를 찾을 수 없습니다. .env 파일을 확인해주세요.")

✅ Gemini API 키가 성공적으로 로드되었습니다.


In [4]:
data_dir = "../data/raw/"
pdf_files = glob.glob(os.path.join(data_dir, "*.pdf"))

print(f"📂 {len(pdf_files)}개의 PDF 문서를 로드합니다...")

documents = []
for file in pdf_files:
    try:
        loader = PyMuPDFLoader(file)
        documents.extend(loader.load())
    except Exception as e:
        print(f"❌ 문서 로드 실패 ({file}): {e}")

print(f"✅ 전체 문서 로드 완료: 총 {len(documents)} 페이지")

# Hasan et al. (2026) 논문 기반 최적 청크 사이즈 적용 (400자)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=40)
splits = text_splitter.split_documents(documents)

print(f"✂️ 문서 분할 완료: 총 {len(splits)} 개의 텍스트 조각(Chunk) 생성")

📂 1개의 PDF 문서를 로드합니다...
✅ 전체 문서 로드 완료: 총 527 페이지
✂️ 문서 분할 완료: 총 2193 개의 텍스트 조각(Chunk) 생성


In [5]:
print("⏳ 한국어 특화 임베딩 모델 로드 중...")
embedding_model = HuggingFaceEmbeddings(model_name="jhgan/ko-sroberta-multitask")

# 단순 조각(Chunk)들을 그대로 DB에 넣습니다 (Parent Document 기법 미사용)
vectorstore = Chroma.from_documents(documents=splits, embedding=embedding_model)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("✅ Baseline Vector DB 구축 완료!")

⏳ 한국어 특화 임베딩 모델 로드 중...


C:\Users\kyle0\AppData\Local\Temp\ipykernel_8196\3940799328.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="jhgan/ko-sroberta-multitask")
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 33167.75it/s]
RobertaModel LOAD REPORT from: jhgan/ko-sroberta-multitask
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Baseline Vector DB 구축 완료!


In [6]:
print("\n" + "="*50)
print("🧐 AI에게 어려운 질문을 던져봅시다 (표 데이터 검색 테스트)")

# 다중 문서 환경의 '벡터 착각'을 피하기 위해 매우 구체적인 프롬프트 작성
query = "삼성전자 2024년 사업보고서에서, 연결재무제표 기준 삼성전자의 당기순이익은 얼마인가요?"
print(f"질문: {query}\n")

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

prompt = ChatPromptTemplate.from_template("""
당신은 꼼꼼한 금융 데이터 분석가입니다.
아래 제공된 [검색된 문서]만을 바탕으로 사용자의 [질문]에 답변해주세요.
표 데이터가 텍스트로 깨져서 숫자의 행/열 기준이 명확하지 않다면, 절대 임의로 숫자를 유추하지 말고 "표 구조 훼손으로 인해 문서에서 명확한 수치를 찾기 어렵습니다."라고 답변하세요.

[검색된 문서]
{context}

[질문]
{input}
""")

document_chain = create_stuff_documents_chain(llm, prompt)
retrieval_chain = create_retrieval_chain(retriever, document_chain)

print("🧠 Gemini가 깨진 텍스트 조각들을 바탕으로 고군분투 중입니다...")
response = retrieval_chain.invoke({"input": query})

print("=" * 50)
print("🤖 최종 AI 답변 (Baseline RAG):")
print(response["answer"])
print("=" * 50)


🧐 AI에게 어려운 질문을 던져봅시다 (표 데이터 검색 테스트)
질문: 삼성전자 2024년 사업보고서에서, 연결재무제표 기준 삼성전자의 당기순이익은 얼마인가요?

🧠 Gemini가 깨진 텍스트 조각들을 바탕으로 고군분투 중입니다...
🤖 최종 AI 답변 (Baseline RAG):
제공된 문서에서는 삼성전자의 2024년 연결재무제표 기준 당기순이익에 대한 정보가 명시되어 있지 않습니다.

문서에는 삼성전자의 2024년 연결기준 매출 301조원, 영업이익 33조원이 언급되어 있으나, 당기순이익은 포함되어 있지 않습니다. 또한, 당기순이익 2,772,012 백만원은 삼성물산㈜의 재무현황으로 명시되어 있습니다.
